# llm_client

> The Claude API client for Phase 10: one structured-output request per turn, an on-disk response cache (the only source of exact reruns), the cap floor, per-call timing records and an SDK retry counter (pricing and the budget are in `llm_budget`), and the git-ignored `.env.local` loader (the offline `FakeClient` is in `llm_fake`). Only this module imports `anthropic`, and only when a live client is built.

In [ ]:
#| default_exp llm_client

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import hashlib
import json
import logging
import os
import threading
import time
from dataclasses import asdict, dataclass
from pathlib import Path

from explRL.llm_budget import PRICES, USAGE, costOf



class RetryCounter(logging.Handler):
    "Counts the SDK's 'Retrying request' log records (429 / 5xx retries) per thread; `install` once per process."
    def __init__(self):
        super().__init__(logging.INFO)
        self.counts = {}

    def emit(self, record):
        if record.getMessage().startswith('Retrying request'):
            self.counts[record.thread] = self.counts.get(record.thread, 0) + 1

    def take(self):
        return self.counts.pop(threading.get_ident(), 0)

    def install(self):
        lg = logging.getLogger('anthropic')
        if self not in lg.handlers:
            lg.addHandler(self)
        if lg.getEffectiveLevel() > logging.INFO:
            lg.setLevel(logging.INFO)


RETRIES = RetryCounter()


def loadEnvFile(path=None):
    """Put KEY=VALUE lines of a git-ignored env file (default: `.env.local` at the repo root) into `os.environ`,
    without overriding variables already set and skipping empty values; returns the names set (never the values)."""
    if path is None:
        root = Path.cwd()
        while not (root / 'pyproject.toml').exists() and root != root.parent:
            root = root.parent
        path = root / '.env.local'
    path, names = Path(path), []
    for line in path.read_text().splitlines() if path.exists() else []:
        k, sep, v = line.strip().removeprefix('export ').partition('=')
        k, v = k.strip(), v.strip().strip('"').strip("'")
        if sep and k and not k.startswith('#') and v and k not in os.environ:
            os.environ[k] = v
            names.append(k)
    return names


def requestKey(req):
    return hashlib.sha256(json.dumps(req, sort_keys=True, separators=(',', ':')).encode()).hexdigest()


@dataclass
class LLMReply:
    text: str
    parsed: object
    stopReason: str
    refusal: object
    usage: dict
    cost: float          # the cost of the original live call (also on cache hits, so rows do not depend on the cache)
    cached: bool = False

    def asDict(self):
        d = asdict(self)
        d.pop('cached')
        return d

    @classmethod
    def fromResponse(cls, resp, price):
        text = next((b.text for b in resp.content if getattr(b, 'type', None) == 'text'), '')
        stop, refusal, parsed = resp.stop_reason, None, None
        if stop == 'refusal':
            refusal = getattr(getattr(resp, 'stop_details', None), 'category', None) or 'unknown'
        elif stop != 'max_tokens':
            try:
                parsed = json.loads(text)
            except (ValueError, TypeError):
                parsed = None
        usage = {k: int(getattr(resp.usage, k, 0) or 0) for k in USAGE}
        return cls(text, parsed, stop, refusal, usage, costOf(usage, price))


class ResponseCache:
    "One JSON file per request key; written atomically; holds replies only (no headers, no credentials)."
    def __init__(self, root):
        self.root = Path(root)

    def _path(self, key):
        return self.root / key[:2] / f'{key[2:]}.json'

    def get(self, key):
        p = self._path(key)
        return json.loads(p.read_text()) if p.exists() else None

    def put(self, key, value):
        "Store unless present (first writer wins, across processes); returns the stored value."
        p = self._path(key)
        p.parent.mkdir(parents=True, exist_ok=True)
        tmp = p.with_name(f'{p.name}.{os.getpid()}.tmp')
        tmp.write_text(json.dumps(value, sort_keys=True))
        try:
            os.link(tmp, p)
            return json.loads(tmp.read_text())
        except FileExistsError:
            return json.loads(p.read_text())
        finally:
            tmp.unlink()


class LLMClient:
    """One request per turn: cached system prompt, one user message, structured-output schema and effort in
    `output_config`; no sampling parameters, no refusal fallback. Replies are cached by request; only live calls are
    charged to the budget."""
    def __init__(self, model, effort='low', maxTokens=2000, cache=None, budget=None, prices=PRICES, client=None,
                 floor=0.0, timing=None):
        if client is None:
            import anthropic                                     # optional extra: pip install -e ".[llm]"
            client = anthropic.Anthropic(max_retries=6)          # credentials from the environment / SDK profile
            RETRIES.install()
        self.model, self.effort, self.maxTokens, self.client = model, effort, int(maxTokens), client
        self.cache, self.budget, self.prices = cache, budget, prices
        self.floor, self.timing, self.tag = float(floor), timing, ''   # floor: dollars kept unspent; tag: the row key
        self.calls = self.hits = 0
        self.pending = 0.0

    def reserve(self, estimate):
        "A run's expected cost: checked against what is left before its first live call (a fully cached rerun is never refused)."
        self.pending = float(estimate)

    def request(self, system, blocks, schema):
        return dict(model=self.model, max_tokens=self.maxTokens,
                    system=[{'type': 'text', 'text': system, 'cache_control': {'type': 'ephemeral'}}],
                    messages=[{'role': 'user', 'content': blocks}],
                    output_config={'effort': self.effort, 'format': {'type': 'json_schema', 'schema': schema}})

    def _record(self, task, t0, cached, retries=0):
        if self.timing is not None:
            self.timing.append(dict(key=self.tag, model=self.model, task=task, seconds=round(time.perf_counter() - t0, 3),
                                    retries=retries, cached=cached))

    def complete(self, system, blocks, schema, task=''):
        t0 = time.perf_counter()
        req = self.request(system, blocks, schema)
        key = requestKey(req)
        hit = self.cache.get(key) if self.cache is not None else None
        if hit is not None:
            self.hits += 1
            self._record(task, t0, True)
            return LLMReply(**hit, cached=True)
        if self.budget is not None:
            self.budget.check(self.pending + self.floor)
            self.pending = 0.0
        RETRIES.take()
        reply = LLMReply.fromResponse(self.client.messages.create(**req), self.prices[self.model])
        self.calls += 1
        if self.budget is not None:         # charge what this call cost
            self.budget.add(reply.cost, self.model)
        if self.cache is not None:          # a concurrent identical request may have stored its reply first: use that
            reply = LLMReply(**self.cache.put(key, reply.asDict()))
        self._record(task, t0, False, RETRIES.take())
        return reply



## Tests

In [ ]:
import json, os, re, tempfile
from pathlib import Path
from explRL.llm_client import *
from explRL.llm_client import requestKey
from explRL.llm_budget import *
from explRL.llm_fake import FakeClient

schema = {'type': 'object', 'properties': {'action': {'type': 'string'}}, 'required': ['action'], 'additionalProperties': False}
tmp = Path(tempfile.mkdtemp())
i0, r0, o0 = TOKEN_PROFILE[('claude-haiku-5-5', 'forage')]
fake = FakeClient([dict(text='{"action": "stay"}', usage=dict(input_tokens=i0, cache_read_input_tokens=r0, output_tokens=o0))])
c = LLMClient('claude-haiku-5-5', cache=ResponseCache(tmp / 'cache'), budget=Budget(1.0, tmp / 'spend.jsonl'), client=fake)
blocks = [{'type': 'text', 'text': 'turn 1'}]
r1 = c.complete('rules', blocks, schema)
assert r1.parsed == {'action': 'stay'} and not r1.cached and c.calls == 1
assert abs(r1.cost - estimateCost('claude-haiku-5-5', 'forage', 1, 1)) < 1e-12   # the profile prices like usage
# a cache hit replays the reply, makes no request and charges nothing
r2 = c.complete('rules', blocks, schema)
assert r2.cached and r2.parsed == r1.parsed and r2.cost == r1.cost and len(fake.requests) == 1 and c.hits == 1
assert abs(c.budget.spent() - r1.cost) < 1e-12
# the request: no sampling parameters, cached system prompt, schema and effort in output_config
req = fake.requests[0]
assert set(req) == {'model', 'max_tokens', 'system', 'messages', 'output_config'}
assert req['system'][0]['cache_control'] == {'type': 'ephemeral'} and req['output_config']['effort'] == 'low'
assert req['output_config']['format'] == {'type': 'json_schema', 'schema': schema}
# first writer wins: a later reply for the same key is not stored, and the stored one is returned
rc = ResponseCache(tmp / 'race')
assert rc.put('ab12', {'x': 1}) == {'x': 1} and rc.put('ab12', {'x': 2}) == {'x': 1} and rc.get('ab12') == {'x': 1}
assert not list((tmp / 'race').rglob('*.tmp'))
# a concurrent identical request that stored its reply first: this call returns that reply but is charged its own cost
rc2, led = ResponseCache(tmp / 'race2'), Budget(10.0, tmp / 'race2.jsonl')
def racer(req):
    rc2.put(requestKey(req), dict(text='{"action": "go"}', parsed={'action': 'go'}, stopReason='end_turn', refusal=None,
                                  usage={}, cost=9.0))
    return dict(text='{"action": "stay"}', usage=dict(input_tokens=1000, output_tokens=0))
r = LLMClient('claude-haiku-5-5', cache=rc2, budget=led, client=FakeClient(racer)).complete('s', blocks, schema)
assert r.parsed == {'action': 'go'} and abs(led.spent() - 1000 * 0.10 / 1e6) < 1e-12
# a different request misses the cache
c.complete('rules', [{'type': 'text', 'text': 'turn 2'}], schema)
assert len(fake.requests) == 2

# refusal and max_tokens stops give no parsed action
for stop, ref in [('refusal', 'cyber'), ('max_tokens', None)]:
    fc = FakeClient([dict(text='{"action": "go"', stop=stop, refusal=ref)])
    r = LLMClient('claude-haiku-5-5', client=fc).complete('s', blocks, schema)
    assert r.parsed is None and r.stopReason == stop and r.refusal == ref
assert LLMClient('claude-haiku-5-5', client=FakeClient(['not json'])).complete('s', blocks, schema).parsed is None

# the cap: a request is refused once the recorded spend reaches it
b = Budget(0.001, tmp / 'cap.jsonl')
cc = LLMClient('claude-haiku-5-5', budget=b, client=FakeClient([dict(text='{}', usage=dict(input_tokens=10000))]))
cc.complete('s', blocks, schema)                    # $0.001: reaches the cap exactly
try:
    cc.complete('s', [{'type': 'text', 'text': 'x'}], schema); raise AssertionError('cap not enforced')
except BudgetExceeded:
    pass
assert len(cc.client.requests) == 1
try:
    Budget(1.0, tmp / 'est.jsonl').check(need=2.0); raise AssertionError
except BudgetExceeded:
    pass
# a run whose estimate exceeds what is left is refused at its first live call, but a cached rerun is served
c.reserve(5.0)
assert c.complete('rules', blocks, schema).cached
try:
    c.complete('rules', [{'type': 'text', 'text': 'new'}], schema); raise AssertionError('estimate not checked')
except BudgetExceeded:
    pass
c.reserve(0.01)
c.complete('rules', [{'type': 'text', 'text': 'new'}], schema)
assert c.pending == 0.0

# costs and estimates
assert costOf(dict(input_tokens=10**6), PRICES['claude-sonnet-5-5']) == 2.0
assert abs(estimateCost('claude-haiku-5-5', 'board', 4, 50, profile={('claude-haiku-5-5', 'board'): (1000, 2000, 300)})
           - 4 * 50 * (1000 * .10 + 2000 * .01 + 300 * .50) / 1e6) < 1e-12
assert requestKey(dict(a=1, b=2)) == requestKey(dict(b=2, a=1))

# secrets: nothing key-like is ever written to the cache or the ledger
written = ''.join(p.read_text() for p in tmp.rglob('*') if p.is_file())
assert not re.search(r'sk-ant-|api[_-]?key|x-api-key|authorization', written, re.I)

# the env-file loader: sets missing, non-empty variables only, and reports names, not values
env = tmp / '.env.local'
env.write_text('# comment\nexport EXPLRL_T1="abc"\nEXPLRL_T2=\nEXPLRL_T3=xyz\nnot a line\n')
os.environ['EXPLRL_T3'] = 'kept'
assert loadEnvFile(env) == ['EXPLRL_T1'] and os.environ['EXPLRL_T1'] == 'abc' and os.environ['EXPLRL_T3'] == 'kept'
assert 'EXPLRL_T2' not in os.environ and loadEnvFile(tmp / 'missing') == []
for k in ('EXPLRL_T1', 'EXPLRL_T3'):
    del os.environ[k]

# the package imports and runs offline without the SDK
import subprocess, sys
code = ("import sys; sys.modules['anthropic'] = None\n"
        "import explRL, explRL.llm_client as m, explRL.llm_fake as f\n"
        "m.LLMClient('claude-haiku-5-5', client=f.FakeClient(['{}'])).complete('s', [], {})\n"
        "try:\n    m.LLMClient('claude-haiku-5-5'); raise SystemExit('imported')\nexcept ImportError: print('ok')")
out = subprocess.run([sys.executable, '-c', code], capture_output=True, text=True)
assert out.stdout.strip() == 'ok', out.stderr

# the floor keeps money unspent: a call is refused when less than the floor would be left
fb = Budget(1.0, tmp / 'floor.jsonl'); fb.add(0.5)
try:
    LLMClient('claude-haiku-5-5', budget=fb, client=FakeClient(['{}']), floor=0.6).complete('s', blocks, schema); raise AssertionError
except BudgetExceeded:
    pass
assert LLMClient('claude-haiku-5-5', budget=fb, client=FakeClient(['{}']), floor=0.4).complete('s', blocks, schema).parsed == {}
# timing records: one per call, live and cached, with the run's tag and the task; no prompt text
tl, tc = [], ResponseCache(tmp / 'timing')
ct = LLMClient('claude-haiku-5-5', cache=tc, client=FakeClient(['{"a": 1}']), timing=tl); ct.tag = 'main|x|001'
ct.complete('s', blocks, schema, task='board'); ct.complete('s', blocks, schema, task='board')
assert [(r['key'], r['task'], r['cached'], r['retries']) for r in tl] == [('main|x|001', 'board', False, 0), ('main|x|001', 'board', True, 0)]
assert set(tl[0]) == {'key', 'model', 'task', 'seconds', 'retries', 'cached'} and tl[0]['seconds'] >= 0
# retries: the SDK's retry log messages are counted for the calling thread
import logging
RETRIES.install(); RETRIES.take()
logging.getLogger('anthropic._base_client').info('Retrying request to /v1/messages in %f seconds', 0.5)
logging.getLogger('anthropic._base_client').info('Sending HTTP Request')
assert RETRIES.take() == 1 and RETRIES.take() == 0


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()